# MATH840 — Challenge 1: Beat your benchmark

**Week 4 | graded assignment 3 of 7 | 8 points | due 23:59 today**

From today you have your own time series. This challenge asks one thing of it: **forecast the next
`h` observations better than the best simple method can**, using the Week 3 toolbox only — no ETS,
no ARIMA. Those are Challenges 2 and 3, and today's number is what makes their improvement visible.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | EDA and visualisation | 1 |
| 2 | Forecast and accuracy | 3 |
| 3 | Justification block | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_ch1_forecast.csv`, `SURNAME_ch1.pdf` and `SURNAME_ch1.ipynb` to the Week 4 activity
on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked.

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from coreforecast.scalers import boxcox, boxcox_lambda

pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier always produces the same series, and this series stays yours for Challenges
2–5. It is real data with its identity removed: no name, no units, values rescaled, the calendar
shifted by whole years, the history trimmed.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)

print(f"series {CODE}: {len(s)} observations, {s['ds'].min().date()} to {s['ds'].max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")
s.tail(3)

### Helpers

Given, and used by the scorer in exactly this form. `mase` is scaled by the in-sample seasonal naive
error of **your whole history**, so the number you compute on your validation window is on the same
scale as the one you will be scored with.

In [ ]:
def benchmark_forecasts(train: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    """The four simple methods from Week 1."""
    T = len(train)
    return {
        "mean":   np.repeat(train.mean(), h),
        "naive":  np.repeat(train[-1], h),
        "snaive": np.array([train[-m + (i % m)] for i in range(h)]),
        "drift":  train[-1] + np.arange(1, h + 1) * (train[-1] - train[0]) / (T - 1),
    }


def mase_scale(history: np.ndarray, m: int) -> float:
    """In-sample seasonal naive MAE - the denominator of MASE, fixed for this series."""
    return float(np.mean(np.abs(history[m:] - history[:-m])))


SCALE = mase_scale(y, M)


def mase(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def future_dates(last: pd.Timestamp, h: int, freq: str) -> pd.DatetimeIndex:
    """The h dates you are forecasting - the scorer expects exactly these."""
    return pd.date_range(last, periods=h + 1, freq=freq)[1:]


FUTURE = future_dates(s["ds"].iloc[-1], H, FREQ)
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())

## 1. EDA and visualisation

*1 point.* This is a series you have never seen. Plot it, and say what you find — the plots are
here to justify the method you pick in Section 2, not to fill space.

- a time plot, always;
- whichever of the seasonal, subseries and ACF plots your argument actually uses;
- one sentence per plot. A plot with no reading attached earns nothing.

In [ ]:
# TODO: plot your series and read it

**What the series shows.**

→

## 2. Forecast and accuracy

*3 points, from the score itself.* Beat the benchmark and you have 3; within 10% of it, 2; worse
than that, 1 and an explanation in Section 3.

### 2.1 Your benchmark

Given. The best of the four simple methods, chosen on a validation window — the last `H`
observations of your history — then refitted on everything. This is the number to beat, and the
scorer recomputes it exactly this way.

In [ ]:
train, valid = y[:-H], y[-H:]

valid_scores = {name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                for name, fc in benchmark_forecasts(train, H, M).items()}
BENCHMARK = min(valid_scores, key=lambda k: valid_scores[k]["rmse"])

print(pd.DataFrame(valid_scores).T.round(3).to_string())
print(f"\nyour benchmark: {BENCHMARK}")

# The benchmark's own forecast of the hidden future, refitted on the whole history.
BENCH_FUTURE = benchmark_forecasts(y, H, M)[BENCHMARK]
BENCH_VALID_MASE = valid_scores[BENCHMARK]["mase"]
print(f"benchmark MASE on the validation window: {BENCH_VALID_MASE:.3f}")

### 2.2 Your model

Build something better, using Week 3 material only. What is available, and all of it is legitimate:

- **a transformation** — Box-Cox, if the seasonal swing grows with the level;
- **forecasting with decomposition** — STL the series, forecast the seasonally adjusted part with
  `naive` or `drift`, carry the seasonal component forward with `snaive`, add them back;
- **a combination** — the mean of two or three forecasts is often better than any one of them;
- **residual diagnostics** — to catch a model that is leaving structure behind.

Judge every candidate on the **same validation window** the benchmark was judged on, with `mase`.
Fit on `train`, score on `valid`, and keep a table of what you tried: Section 3 asks for it.

In [ ]:
# TODO: build your candidates, score them on the validation window, keep the table.
#
# A decomposition forecast, for example, is: STL(train) -> seasonally adjusted = train - seasonal,
# forecast that with naive or drift, then add the seasonal component of the matching season.

**Your candidates and their validation scores.**

→

### 2.3 Your submitted forecast

Refit your chosen approach on the **whole** history `y` and forecast `H` steps. Then fill
`FORECAST`: `H` rows, in date order, with an 80% interval.

An interval is not marked this week, but it must be there and it must not be nonsense. The crude
honest version: the standard deviation of your one-step residuals, widened by $\sqrt{k}$ at step
$k$, times 1.28.

In [ ]:
# TODO: your final forecast on the full history.
YHAT = None          # np.array of length H
YHAT_LO_80 = None    # np.array of length H
YHAT_HI_80 = None    # np.array of length H

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H:
        raise ValueError(f"{name} must be an array of length H = {H}")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Be specific: numbers from
your own cells, not adjectives.

**3.1 What the data shows** *(0.75)* — frequency, seasonal period, trend, level shifts, outliers,
each claim tied to a plot from Section 1.

→

**3.2 Why this method** *(0.75)* — what you submitted, why it fits what you just described, and
what you tried and dropped.

→

**3.3 What the residuals say** *(0.75)* — the ACF and a Ljung-Box test on your final model's
residuals, read: what structure is left, and whether it matters at this horizon.

→

In [ ]:
# TODO: residual diagnostics of your final model (ACF + Ljung-Box)

**3.4 What you expect** *(0.75)* — your validation MASE against the benchmark's, and the number you
expect on the hidden holdout. If you expect to lose, say what the holdout may contain that your
validation window did not.

→

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: everything is fitted on `y` or on `train`, never on anything after them.
- Every number in your text comes from a cell.

## 5. Submit

Run both cells, then upload **three files** to the Week 4 activity on Moodle before 23:59:

- `SURNAME_ch1_forecast.csv` — written below;
- `SURNAME_ch1.pdf` — `File → Print → Save as PDF`;
- `SURNAME_ch1.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_ch1_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks the scorer runs. If this cell passes, your file can be scored.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")